# Outlier Detection — Python Code

This notebook mirrors the workshop presentation. Each section is deliberately kept short.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Create the dataset

In [ ]:
names   = ['Alice','Bob','Carol','David','Eve','Frank','Grace','Henry',
           'Iris','Jack','Karen','Leo','Maria','Nick','Olivia','Paul',
           'Quinn','Rachel','Steve','Tom']

incomes = [800,2800,3100,3250,3400,3450,3500,3550,3600,3650,
           3700,3750,3800,3850,3900,4000,4100,4200,4350,9500]

df = pd.DataFrame({'customer': names, 'income': incomes})
print(df.to_string(index=False))

## 2. Descriptive statistics

In [ ]:
print(df['income'].describe().round(2))

## 3. Distribution plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors  = ['red' if v == 800 else 'orange' if v == 9500 else 'steelblue' for v in sorted(incomes)]
ax.bar(range(20), sorted(incomes), color=colors, edgecolor='white')
ax.set_xticks(range(20))
ax.set_xticklabels([f'${v:,}' for v in sorted(incomes)], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Monthly Income ($)')
ax.set_title('20 Customer Incomes — Sorted')
red_p = mpatches.Patch(color='red',    label='Low outlier')
ora_p = mpatches.Patch(color='orange', label='High outlier')
blu_p = mpatches.Patch(color='steelblue', label='Normal')
ax.legend(handles=[blu_p, red_p, ora_p])
plt.tight_layout()
plt.show()

## 4. IQR calculation

In [ ]:
Q1    = df['income'].quantile(0.25)
Q2    = df['income'].quantile(0.50)
Q3    = df['income'].quantile(0.75)
IQR   = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Q1  (25th pct)  : ${Q1:,.2f}')
print(f'Q2  (50th pct)  : ${Q2:,.2f}')
print(f'Q3  (75th pct)  : ${Q3:,.2f}')
print(f'IQR (Q3 - Q1)   : ${IQR:,.2f}')
print(f'Lower fence     : ${lower:,.2f}')
print(f'Upper fence     : ${upper:,.2f}')

## 5. Identify outliers

In [ ]:
outliers = df[(df['income'] < lower) | (df['income'] > upper)]
print(outliers)

## 6. Boxplot — before handling

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.boxplot(df['income'], vert=False, patch_artist=True,
           boxprops=dict(facecolor='#DBEAFE', color='steelblue'),
           medianprops=dict(color='navy', linewidth=2),
           flierprops=dict(marker='o', markerfacecolor='red', markersize=10))
ax.set_xlabel('Monthly Income ($)')
ax.set_title('Boxplot — Before Handling Outliers')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 7. Cap the outliers and recalculate

In [ ]:
df['income_capped'] = df['income'].clip(lower=lower, upper=upper)

Q1c  = df['income_capped'].quantile(0.25)
Q3c  = df['income_capped'].quantile(0.75)
IQRc = Q3c - Q1c

print('After capping:')
print(f'Mean   : ${df["income_capped"].mean():,.2f}  (was ${df["income"].mean():,.2f})')
print(f'Median : ${df["income_capped"].median():,.2f}  (was ${df["income"].median():,.2f})')
print(f'Q1     : ${Q1c:,.2f}')
print(f'Q3     : ${Q3c:,.2f}')
print(f'IQR    : ${IQRc:,.2f}  (was ${IQR:,.2f})')

## 8. Boxplot — before vs after

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

for ax, col, title, color in [
    (axes[0], 'income',        'Before — with outliers', '#FCA5A5'),
    (axes[1], 'income_capped', 'After  — capped',        '#86EFAC'),
]:
    ax.boxplot(df[col], vert=False, patch_artist=True,
               boxprops=dict(facecolor=color, color='gray'),
               medianprops=dict(color='navy', linewidth=2),
               flierprops=dict(marker='o', markerfacecolor='red', markersize=8))
    ax.set_title(title)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.set_xlim(0, 10000)

plt.tight_layout()
plt.show()